In [1]:
# Install and import required packages
!pip install -U scikit-learn imbalanced-learn optuna

# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import optuna

# Data preprocessing
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils import resample
from sklearn.model_selection import cross_val_score

# Feature selection
from sklearn.feature_selection import RFECV

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Metrics and evaluation
from sklearn.metrics import (roc_auc_score, classification_report, 
                           confusion_matrix, roc_curve, auc, 
                           precision_recall_curve, f1_score)

# Visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.0 MB 910.3 kB/s eta 0:00:09
   --- ------------------------------------ 0.8/8.0 MB 894.4 kB/s eta 0:00:09
   ----- ---------------------------------- 1.0/8.0 MB 1.0 MB/s eta 0:00:07
   ------ --------------------------------- 1.3/8.0 MB 1.1 MB/s eta 0:00:07
   ------- -------------------------------- 1.6/8.0 MB 1.1 MB/s eta 0:00:06
   ------- -------------------------------- 1.6/8.0 MB 1.1 MB/s eta 0:00:06
   --------- ------------------------------ 1.8/8.0 MB 1.0 MB/s eta 0:00:07
   --------- ------------------------------ 1.8/8.0 MB 1.0 MB/s eta 0:00:07
   ---------- ----------------------------- 2.1/8.0 MB 856.8 kB/s eta 0:00:07
   ---------- --------------------

# 2. Exploratory Data Analysis (EDA): 
**2.1 Initial Data Inspection** :

First we load and inspect the dataset structure and basic statistics

In [22]:
import pandas as pd

df = pd.read_csv("../data/cs-training.csv", sep=";")
df.drop(columns=["Unnamed: 0"], inplace=True)
print(df.shape)
df.head()
display(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nDescriptive stats:")
print(df.describe())
print("\nMissing Values Count:")
print(df.isnull().sum())
print("\nDuplicated Values Count:")
print(df.duplicated().sum())

(150000, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0



Data Types:
SeriousDlqin2yrs                          int64
RevolvingUtilizationOfUnsecuredLines    float64
age                                       int64
NumberOfTime30-59DaysPastDueNotWorse      int64
DebtRatio                               float64
MonthlyIncome                           float64
NumberOfOpenCreditLinesAndLoans           int64
NumberOfTimes90DaysLate                   int64
NumberRealEstateLoansOrLines              int64
NumberOfTime60-89DaysPastDueNotWorse      int64
NumberOfDependents                      float64
dtype: object

Descriptive stats:
       SeriousDlqin2yrs  RevolvingUtilizationOfUnsecuredLines            age  \
count     150000.000000                         150000.000000  150000.000000   
mean           0.066840                              6.048438      52.295207   
std            0.249746                            249.755371      14.771866   
min            0.000000                              0.000000       0.000000   
25%            0.000000  

# Initial Data Exploration: 
**dataset overview**  : The dataset contains 150,000 entries and 11 columns (after removing the index column Unnamed: 0). It is originally from the Kaggle challenge "Give Me Some Credit" and is used to predict whether a borrower will experience financial distress in the next two years.

**First Look**: Displaying the first 5 rows provides a glimpse into the data structure, showing a mix of behavioral, financial, and personal features for each borrower.

**Data Types**: The dataset includes numerical features only — a mix of integers (age, late payment counts, number of loans) and floats (income, debt ratio, credit utilization). The target variable SeriousDlqin2yrs is binary: 1 means the borrower experienced serious delinquency, 0 means they did not.

**Missing Values**: this dataset does contain missing values in 2 columns — MonthlyIncome has 29,731 missing values (19.8%) and NumberOfDependents has 3,924 missing values (2.6%). 